
# QPLIB B&B Dataset Generation

This notebook is **only for QPLIB**.

It starts from the final 143 fully bounded QPLIB pickles and generates the raw B&B dataset needed before the later learned time-budget / relaxation-selection stages.

Pipeline:

1. Load and validate the final QPLIB bounded pickles.
2. Find a verified feasible incumbent for each instance:
   - multi-start Ipopt,
   - manual feasibility check,
   - Gurobi feasibility-repair fallback if needed.
3. Run two separate fixed-budget B&B trees:
   - LP tree,
   - SDP′ tree.
4. Preserve the existing thesis B&B rules:
   - BFS,
   - sequential branching,
   - midpoint split,
   - common incumbent for LP and SDP′.
5. Save:
   - feasible incumbent CSV/pickle,
   - node-level LIVE CSV/parquet,
   - per-instance B&B summary.

This notebook does **not** run MINLPLib.


In [3]:

# ============================================================
# 0. CONFIG + IMPORTS
# ============================================================

from pathlib import Path
from dataclasses import dataclass, field
from collections import deque
from typing import Any, Dict, List, Optional, Tuple

import os
import sys
import gc
import math
import time
import pickle
import traceback
import importlib
import inspect

import numpy as np
import pandas as pd

BASE = Path(r"C:\Users\suuser\Desktop\minlplib")
QPLIB_BASE = BASE / "Areview" / "qplib"

# Final QPLIB dataset already prepared earlier.
QPLIB_INPUT_DIR = QPLIB_BASE / "pickles_nlt300_mge1_bounded"

# New outputs for the QPLIB B&B extension.
QPLIB_BB_DIR = QPLIB_BASE / "bb_qplib"
QPLIB_BB_DIR.mkdir(parents=True, exist_ok=True)

QPLIB_FEASIBLE_CSV = QPLIB_BB_DIR / "qplib_feasible_ubs.csv"
QPLIB_FEASIBLE_PKL = QPLIB_BB_DIR / "qplib_feasible_points.pkl"

QPLIB_LIVE_CSV = QPLIB_BB_DIR / "qplib_bb_fixed100_LIVE.csv"
QPLIB_LIVE_PARQUET = QPLIB_BB_DIR / "qplib_bb_fixed100_LIVE.parquet"
QPLIB_BB_SUMMARY_CSV = QPLIB_BB_DIR / "qplib_bb_fixed100_instance_summary.csv"
QPLIB_BB_SKIPPED_CSV = QPLIB_BB_DIR / "qplib_bb_fixed100_SKIPPED.csv"

# Current verified Fusion/Gurobi QCQP solver.
SOLVER_MODULE = "qcqp_solverminlplib"

# ------------------------------------------------------------
# QPLIB incumbent search
# ------------------------------------------------------------
RUN_IPOPT = True
IPOPT_TIME_LIMIT_PER_START = 20.0
IPOPT_MAX_ITER = 1000
IPOPT_TOL = 1e-7
IPOPT_CONSTR_VIOL_TOL = 1e-7

N_RANDOM_STARTS = 5
RANDOM_SEED = 123
RANDOM_RADIUS_CAP = 100.0

RUN_GUROBI_REPAIR_FALLBACK = True
GUROBI_REPAIR_TIME_LIMIT = 60.0

FEAS_TOL = 1e-6
BOUND_TOL = 1e-7

# Resume incumbent search if output already exists.
RESUME_FEASIBLE_SEARCH = True

# ------------------------------------------------------------
# QPLIB first B&B data-generation run
# ------------------------------------------------------------
FIXED_NODES_LP = 100
FIXED_NODES_SDPP = 100
MAX_DEPTH = 999
MIN_WIDTH = 1e-10
PRUNE_TOL = 1e-8

BRANCH_RULE = "sequential"
SPLIT_RULE = "midpoint"

USE_BLOCKS = True
BLOCK_TOL = 1e-8
VERBOSE_BLOCKS = False
PRUNE_MATRICES = False
MIN_BLOCK_EDGES = 0

FLUSH_EVERY_NODES = 5
RESUME_BB = True

# For quick testing, e.g. 5. Use None for all 143.
MAX_QPLIB_INSTANCES = None

# ------------------------------------------------------------
# Target definition used in the paper
# ------------------------------------------------------------
ETA = 1e-3
EPS = 1e-2

print("QPLIB input dir:", QPLIB_INPUT_DIR)
print("QPLIB B&B output dir:", QPLIB_BB_DIR)


QPLIB input dir: C:\Users\suuser\Desktop\minlplib\Areview\qplib\pickles_nlt300_mge1_bounded
QPLIB B&B output dir: C:\Users\suuser\Desktop\minlplib\Areview\qplib\bb_qplib


In [4]:

# ============================================================
# 1. LOAD CURRENT SOLVER + OPTIONAL INCUMBENT SOLVERS
# ============================================================

if str(BASE) not in sys.path:
    sys.path.insert(0, str(BASE))

solver_module = importlib.import_module(SOLVER_MODULE)
solver_module = importlib.reload(solver_module)
QCQPSolver = solver_module.QCQPSolver

print("Solver file:", solver_module.__file__)
print("QCQPSolver signature:", inspect.signature(QCQPSolver.__init__))

try:
    import cyipopt
    HAS_CYIPOPT = True
except Exception as e:
    cyipopt = None
    HAS_CYIPOPT = False
    print("[WARN] cyipopt unavailable:", repr(e))

try:
    import gurobipy as gp
    from gurobipy import GRB
    HAS_GUROBI = True
except Exception as e:
    gp = None
    GRB = None
    HAS_GUROBI = False
    print("[WARN] gurobipy unavailable:", repr(e))

print("HAS_CYIPOPT:", HAS_CYIPOPT)
print("HAS_GUROBI :", HAS_GUROBI)


Solver file: C:\Users\suuser\Desktop\minlplib\qcqp_solverminlplib.py
QCQPSolver signature: (self, A_list, b_list, c_list, l, u, use_blocks=True, block_tol=1e-08, verbose_blocks=True, prune_matrices=False, min_block_edges=0)
HAS_CYIPOPT: True
HAS_GUROBI : True


In [5]:

# ============================================================
# 2. QPLIB DATA HELPERS
# Canonical convention:
#     0.5 x^T A x + b^T x + c
# ============================================================

def _get(obj, key, default=None):
    if isinstance(obj, dict):
        return obj.get(key, default)
    return getattr(obj, key, default)


def _to_dict(obj):
    if isinstance(obj, dict):
        return dict(obj)
    if hasattr(obj, "__dict__"):
        return dict(obj.__dict__)
    raise TypeError(f"Unsupported pickle object type: {type(obj).__name__}")


def load_qplib_pickle(pkl_path: Path) -> Dict[str, Any]:
    with open(pkl_path, "rb") as f:
        obj = pickle.load(f)

    d = _to_dict(obj)

    A_list = [np.asarray(A, dtype=float) for A in d["A_list"]]
    b_list = [np.asarray(b, dtype=float).reshape(-1) for b in d["b_list"]]
    c_list = [float(np.asarray(c, dtype=float).reshape(-1)[0]) for c in d["c_list"]]

    # Final Stage-5 files may contain l_work/u_work as well as l/u.
    # Prefer working bounds if they exist and are finite.
    l_work = d.get("l_work", None)
    u_work = d.get("u_work", None)

    if l_work is not None and u_work is not None:
        l_try = np.asarray(l_work, dtype=float).reshape(-1)
        u_try = np.asarray(u_work, dtype=float).reshape(-1)
        if np.all(np.isfinite(l_try)) and np.all(np.isfinite(u_try)):
            l, u = l_try, u_try
        else:
            l = np.asarray(d["l"], dtype=float).reshape(-1)
            u = np.asarray(d["u"], dtype=float).reshape(-1)
    else:
        l = np.asarray(d["l"], dtype=float).reshape(-1)
        u = np.asarray(d["u"], dtype=float).reshape(-1)

    n = len(l)

    if len(A_list) != len(b_list) or len(A_list) != len(c_list):
        raise ValueError("A_list, b_list, c_list lengths disagree.")

    for k, (A, b) in enumerate(zip(A_list, b_list)):
        if A.shape != (n, n):
            raise ValueError(f"A[{k}] shape {A.shape}, expected {(n,n)}.")
        if b.shape != (n,):
            raise ValueError(f"b[{k}] shape {b.shape}, expected {(n,)}.")

    if not np.all(np.isfinite(l)) or not np.all(np.isfinite(u)):
        raise ValueError("Final QPLIB B&B instance still has non-finite bounds.")
    if np.any(l > u):
        raise ValueError("Invalid bounds: some lower bound exceeds upper bound.")

    name = str(
        d.get("name")
        or d.get("instance_name")
        or d.get("problem_name")
        or pkl_path.stem
    )

    return {
        "file": str(pkl_path),
        "name": name,
        "pkl_stem": pkl_path.stem,
        "instance_id": d.get("instance_id", d.get("id", name)),
        "probtype": d.get("probtype"),
        "integrality_relaxed": d.get("integrality_relaxed"),
        "convex": d.get("convex", d.get("qplib_convex")),
        "n": n,
        "m_original": d.get("m_original"),
        "m_solver": len(A_list) - 1,
        "A_list": A_list,
        "b_list": b_list,
        "c_list": c_list,
        "l": l,
        "u": u,
        "raw": d,
    }


def sym_A(A):
    A = np.asarray(A, dtype=float)
    return 0.5 * (A + A.T)


def q_value(A, b, c, x):
    """
    QPLIB/current-solver canonical form:
        0.5*x^T A x + b^T x + c

    Using sym_A is value-equivalent for real x and gives the correct gradient.
    """
    x = np.asarray(x, dtype=float)
    S = sym_A(A)
    return float(0.5 * x @ S @ x + np.asarray(b, dtype=float) @ x + float(c))


def q_grad(A, b, x):
    x = np.asarray(x, dtype=float)
    return sym_A(A) @ x + np.asarray(b, dtype=float)


def objective_value(data, x):
    return q_value(data["A_list"][0], data["b_list"][0], data["c_list"][0], x)


def constraint_values(data, x):
    vals = [
        q_value(data["A_list"][k], data["b_list"][k], data["c_list"][k], x)
        for k in range(1, len(data["A_list"]))
    ]
    return np.asarray(vals, dtype=float)


def check_feasibility(data, x, feas_tol=FEAS_TOL, bound_tol=BOUND_TOL):
    x = np.asarray(x, dtype=float).reshape(-1)
    l = data["l"]
    u = data["u"]

    lower_viol = np.maximum(l - x, 0.0)
    upper_viol = np.maximum(x - u, 0.0)
    max_bound_violation = float(
        max(
            np.max(lower_viol) if lower_viol.size else 0.0,
            np.max(upper_viol) if upper_viol.size else 0.0,
        )
    )

    g = constraint_values(data, x)
    max_constraint_violation = float(
        np.max(np.maximum(g, 0.0)) if g.size else 0.0
    )

    return {
        "is_feasible": bool(
            max_bound_violation <= bound_tol
            and max_constraint_violation <= feas_tol
        ),
        "max_bound_violation": max_bound_violation,
        "max_constraint_violation": max_constraint_violation,
    }


pkl_files = sorted(QPLIB_INPUT_DIR.glob("*.pkl"))
if MAX_QPLIB_INSTANCES is not None:
    pkl_files = pkl_files[:MAX_QPLIB_INSTANCES]

print("QPLIB B&B input pickles:", len(pkl_files))

validation_rows = []
for p in pkl_files:
    try:
        d = load_qplib_pickle(p)
        validation_rows.append({
            "name": d["name"],
            "file": p.name,
            "n": d["n"],
            "m_solver": d["m_solver"],
            "probtype": d["probtype"],
            "integrality_relaxed": d["integrality_relaxed"],
            "convex": d["convex"],
            "valid": True,
            "reason": "",
        })
    except Exception as e:
        validation_rows.append({
            "name": p.stem,
            "file": p.name,
            "valid": False,
            "reason": repr(e),
        })

validation_df = pd.DataFrame(validation_rows)
display(validation_df.head())
print("Valid:", int(validation_df["valid"].sum()), "/", len(validation_df))


QPLIB B&B input pickles: 143


,name,file,n,m_solver,probtype,integrality_relaxed,convex,valid,reason
0,QPLIB_0018,QPLIB_0018.pkl,50,2,QCL,True,None,True,
1,QPLIB_0031,QPLIB_0031.pkl,60,33,QML,True,None,True,
2,QPLIB_0032,QPLIB_0032.pkl,100,53,QML,True,None,True,
3,QPLIB_0067,QPLIB_0067.pkl,80,1,QBL,True,None,True,
4,QPLIB_0343,QPLIB_0343.pkl,50,2,QCL,True,None,True,


Valid: 143 / 143


In [6]:

# ============================================================
# 3. QPLIB FEASIBLE INCUMBENT SEARCH
# Same high-level strategy as the MINLPLib B&B notebook:
#   multi-start Ipopt -> manual feasibility check -> Gurobi repair fallback.
# ============================================================

def make_start_points(l, u, n_random=N_RANDOM_STARTS, seed=0):
    rng = np.random.default_rng(seed)

    l = np.asarray(l, dtype=float)
    u = np.asarray(u, dtype=float)
    width = u - l

    starts = []

    mid = 0.5 * (l + u)
    starts.append(("midpoint", mid.copy()))
    starts.append(("clipped_zero", np.clip(np.zeros_like(l), l, u)))
    starts.append(("lower_25pct", l + 0.25 * width))
    starts.append(("upper_75pct", l + 0.75 * width))

    # Same idea as MINLPLib: local random starts so giant boxes do not
    # produce absurd starting values.
    radius = np.minimum(0.25 * width, RANDOM_RADIUS_CAP)

    for r in range(n_random):
        x = mid + rng.uniform(-1.0, 1.0, size=len(l)) * radius
        x = np.clip(x, l, u)
        starts.append((f"random_{r}", x))

    unique = []
    seen = set()
    for name, x in starts:
        key = tuple(np.round(x, decimals=10))
        if key not in seen:
            seen.add(key)
            unique.append((name, x))

    return unique


class IpoptQPLIBProblem:
    def __init__(self, data):
        self.data = data
        self.n = data["n"]
        self.m = data["m_solver"]

    def objective(self, x):
        return objective_value(self.data, x)

    def gradient(self, x):
        return q_grad(
            self.data["A_list"][0],
            self.data["b_list"][0],
            x
        )

    def constraints(self, x):
        return constraint_values(self.data, x)

    def jacobian(self, x):
        rows = []
        for k in range(1, len(self.data["A_list"])):
            rows.append(
                q_grad(
                    self.data["A_list"][k],
                    self.data["b_list"][k],
                    x
                )
            )

        if not rows:
            return np.array([], dtype=float)

        return np.vstack(rows).reshape(-1)


def solve_with_ipopt_qplib(data, starts):
    if not HAS_CYIPOPT or not RUN_IPOPT:
        return {
            "best_x": None,
            "best_ub": np.inf,
            "best_start": None,
            "attempts": 0,
            "status_summary": "cyipopt_not_available_or_disabled",
            "runtime": 0.0,
        }

    n = data["n"]
    m = data["m_solver"]

    # All canonicalized QPLIB constraints are <= 0.
    cl = -1e19 * np.ones(m, dtype=float)
    cu = np.zeros(m, dtype=float)

    best_x = None
    best_ub = np.inf
    best_start = None
    statuses = []
    attempts = 0

    t0_all = time.time()

    for start_name, x0 in starts:
        attempts += 1

        problem_obj = IpoptQPLIBProblem(data)

        nlp = cyipopt.Problem(
            n=n,
            m=m,
            problem_obj=problem_obj,
            lb=data["l"],
            ub=data["u"],
            cl=cl,
            cu=cu,
        )

        nlp.add_option("print_level", 0)
        nlp.add_option("max_iter", int(IPOPT_MAX_ITER))
        nlp.add_option("tol", float(IPOPT_TOL))
        nlp.add_option("constr_viol_tol", float(IPOPT_CONSTR_VIOL_TOL))
        nlp.add_option("max_cpu_time", float(IPOPT_TIME_LIMIT_PER_START))
        nlp.add_option("hessian_approximation", "limited-memory")
        nlp.add_option("mu_strategy", "adaptive")

        t0 = time.time()

        try:
            x_sol, info = nlp.solve(np.asarray(x0, dtype=float))
            runtime = time.time() - t0

            status = info.get("status", None)
            obj_ipopt = float(info.get("obj_val", np.nan))
            check = check_feasibility(data, x_sol)

            statuses.append(
                f"{start_name}:status={status},obj={obj_ipopt:.8g},"
                f"feas={check['is_feasible']},"
                f"viol={check['max_constraint_violation']:.2e},"
                f"t={runtime:.2f}"
            )

            # Important: accept ONLY the result of our own manual check.
            if check["is_feasible"]:
                ub = objective_value(data, x_sol)
                if ub < best_ub:
                    best_ub = float(ub)
                    best_x = np.asarray(x_sol, dtype=float)
                    best_start = start_name

        except Exception as e:
            runtime = time.time() - t0
            statuses.append(
                f"{start_name}:error={type(e).__name__}:{str(e)},t={runtime:.2f}"
            )

    return {
        "best_x": best_x,
        "best_ub": best_ub,
        "best_start": best_start,
        "attempts": attempts,
        "status_summary": " | ".join(statuses)[:4000],
        "runtime": time.time() - t0_all,
    }


def add_qexpr_gurobi_qplib(x_vars, A, b, c, tol=0.0):
    """
    Build:
        0.5*x^T A*x + b^T*x + c

    We loop only over nonzero entries for speed.
    """
    expr = gp.QuadExpr()
    expr.add(float(c))

    b = np.asarray(b, dtype=float)
    for i in np.where(np.abs(b) > tol)[0]:
        expr.add(float(b[i]) * x_vars[int(i)])

    A = np.asarray(A, dtype=float)
    nz = np.argwhere(np.abs(A) > tol)
    for i, j in nz:
        expr.add(0.5 * float(A[i, j]) * x_vars[int(i)] * x_vars[int(j)])

    return expr


def solve_gurobi_repair_qplib(data):
    if not HAS_GUROBI or not RUN_GUROBI_REPAIR_FALLBACK:
        return {
            "has_solution": False,
            "x": None,
            "ub": np.inf,
            "status": "gurobi_not_available_or_disabled",
            "slack_sum": np.inf,
            "runtime": 0.0,
        }

    n = data["n"]
    m = data["m_solver"]

    model = gp.Model(f"qplib_repair_{data['name']}")
    model.Params.OutputFlag = 0
    model.Params.TimeLimit = float(GUROBI_REPAIR_TIME_LIMIT)
    model.Params.NonConvex = 2

    x = [
        model.addVar(
            lb=float(data["l"][i]),
            ub=float(data["u"][i]),
            vtype=GRB.CONTINUOUS,
            name=f"x[{i}]",
        )
        for i in range(n)
    ]

    s = [
        model.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name=f"s[{k}]")
        for k in range(1, m + 1)
    ]

    model.update()

    for k in range(1, m + 1):
        qk = add_qexpr_gurobi_qplib(
            x,
            data["A_list"][k],
            data["b_list"][k],
            data["c_list"][k],
        )
        model.addQConstr(qk <= s[k - 1], name=f"repair_qc[{k}]")

    model.setObjective(gp.quicksum(s), GRB.MINIMIZE)

    t0 = time.time()
    model.optimize()
    runtime = time.time() - t0

    if model.SolCount == 0:
        return {
            "has_solution": False,
            "x": None,
            "ub": np.inf,
            "status": f"gurobi_status_{model.Status}",
            "slack_sum": np.inf,
            "runtime": runtime,
        }

    x_val = np.array([var.X for var in x], dtype=float)
    slack_sum = float(sum(sk.X for sk in s))

    check = check_feasibility(data, x_val)
    ub = objective_value(data, x_val) if check["is_feasible"] else np.inf

    return {
        "has_solution": True,
        "x": x_val,
        "ub": float(ub),
        "status": f"gurobi_status_{model.Status}",
        "slack_sum": slack_sum,
        "runtime": runtime,
        "check": check,
    }


def load_existing_feasible_points():
    points = {}
    old_df = pd.DataFrame()

    if RESUME_FEASIBLE_SEARCH and QPLIB_FEASIBLE_PKL.exists():
        try:
            with open(QPLIB_FEASIBLE_PKL, "rb") as f:
                points = pickle.load(f)
            if not isinstance(points, dict):
                points = {}
        except Exception:
            points = {}

    if RESUME_FEASIBLE_SEARCH and QPLIB_FEASIBLE_CSV.exists():
        try:
            old_df = pd.read_csv(QPLIB_FEASIBLE_CSV)
        except Exception:
            old_df = pd.DataFrame()

    return points, old_df


def run_qplib_feasible_search(pkl_files):
    feasible_points, old_df = load_existing_feasible_points()

    completed = set()
    if not old_df.empty and "name" in old_df.columns:
        completed = set(old_df["name"].astype(str))

    rows = old_df.to_dict("records") if not old_df.empty else []

    for idx, pkl_path in enumerate(pkl_files):
        data = load_qplib_pickle(pkl_path)
        name = str(data["name"])

        if RESUME_FEASIBLE_SEARCH and name in completed:
            print(f"[{idx+1}/{len(pkl_files)}] {name}: already processed -> skip")
            continue

        print(f"\n[{idx+1}/{len(pkl_files)}] {name} | n={data['n']} m={data['m_solver']}")

        starts = make_start_points(
            data["l"],
            data["u"],
            n_random=N_RANDOM_STARTS,
            seed=RANDOM_SEED + idx,
        )

        ipopt_res = solve_with_ipopt_qplib(data, starts)

        chosen_x = None
        chosen_ub = np.inf
        chosen_source = None
        repair_res = None

        if ipopt_res["best_x"] is not None and np.isfinite(ipopt_res["best_ub"]):
            chosen_x = np.asarray(ipopt_res["best_x"], dtype=float)
            chosen_ub = float(ipopt_res["best_ub"])
            chosen_source = f"ipopt:{ipopt_res['best_start']}"
        elif RUN_GUROBI_REPAIR_FALLBACK:
            repair_res = solve_gurobi_repair_qplib(data)
            if (
                repair_res.get("x") is not None
                and np.isfinite(repair_res.get("ub", np.inf))
            ):
                chosen_x = np.asarray(repair_res["x"], dtype=float)
                chosen_ub = float(repair_res["ub"])
                chosen_source = "gurobi_repair"

        if chosen_x is not None:
            check = check_feasibility(data, chosen_x)
            accepted = bool(check["is_feasible"])
        else:
            check = {
                "is_feasible": False,
                "max_bound_violation": np.nan,
                "max_constraint_violation": np.nan,
            }
            accepted = False

        row = {
            "name": name,
            "file": pkl_path.name,
            "probtype": data["probtype"],
            "n": data["n"],
            "m_solver": data["m_solver"],
            "integrality_relaxed": data["integrality_relaxed"],
            "accepted": accepted,
            "ub": chosen_ub if accepted else np.nan,
            "source": chosen_source,
            "max_bound_violation": check["max_bound_violation"],
            "max_constraint_violation": check["max_constraint_violation"],
            "ipopt_attempts": ipopt_res["attempts"],
            "ipopt_runtime": ipopt_res["runtime"],
            "ipopt_status_summary": ipopt_res["status_summary"],
            "gurobi_repair_status": (
                repair_res.get("status") if repair_res is not None else None
            ),
            "gurobi_repair_slack_sum": (
                repair_res.get("slack_sum") if repair_res is not None else np.nan
            ),
            "gurobi_repair_runtime": (
                repair_res.get("runtime") if repair_res is not None else 0.0
            ),
        }

        rows.append(row)

        if accepted:
            feasible_points[name] = {
                "name": name,
                "file": pkl_path.name,
                "x": chosen_x,
                "ub": float(chosen_ub),
                "source": chosen_source,
                "max_bound_violation": check["max_bound_violation"],
                "max_constraint_violation": check["max_constraint_violation"],
            }

        # Checkpoint after EVERY instance.
        pd.DataFrame(rows).drop_duplicates(
            subset=["name"], keep="last"
        ).to_csv(QPLIB_FEASIBLE_CSV, index=False)

        with open(QPLIB_FEASIBLE_PKL, "wb") as f:
            pickle.dump(feasible_points, f)

        print(
            "accepted =", accepted,
            "| source =", chosen_source,
            "| UB =", chosen_ub if accepted else None,
        )

        gc.collect()

    final_df = pd.DataFrame(rows).drop_duplicates(
        subset=["name"], keep="last"
    ).reset_index(drop=True)

    final_df.to_csv(QPLIB_FEASIBLE_CSV, index=False)
    with open(QPLIB_FEASIBLE_PKL, "wb") as f:
        pickle.dump(feasible_points, f)

    print("\nFeasible incumbent summary")
    print(final_df["accepted"].value_counts(dropna=False))
    print("Saved:", QPLIB_FEASIBLE_CSV)
    print("Saved:", QPLIB_FEASIBLE_PKL)

    return final_df, feasible_points


# RUN THIS CELL TO GENERATE / RESUME FEASIBLE INCUMBENTS.
qplib_feasible_df, qplib_feasible_points = run_qplib_feasible_search(pkl_files)

display(qplib_feasible_df.head())



[1/143] QPLIB_0018 | n=50 m=2
accepted = True | source = ipopt:clipped_zero | UB = -6.386018862109888

[2/143] QPLIB_0031 | n=60 m=33
accepted = True | source = ipopt:clipped_zero | UB = 15.20052472562084

[3/143] QPLIB_0032 | n=100 m=53
accepted = True | source = ipopt:clipped_zero | UB = 9.94558798396219

[4/143] QPLIB_0067 | n=80 m=1
accepted = True | source = ipopt:clipped_zero | UB = -112205.36058245547

[5/143] QPLIB_0343 | n=50 m=2
accepted = True | source = ipopt:clipped_zero | UB = -6.386018862109817

[6/143] QPLIB_0633 | n=75 m=2
accepted = True | source = ipopt:random_0 | UB = 79.57916821078919

[7/143] QPLIB_0681 | n=215 m=538
accepted = True | source = ipopt:random_0 | UB = -48.26342933984606

[8/143] QPLIB_0682 | n=261 m=716
accepted = True | source = ipopt:random_2 | UB = -48.04304402260596

[9/143] QPLIB_0698 | n=118 m=157
accepted = True | source = ipopt:random_3 | UB = 968572.1425647809

[10/143] QPLIB_0752 | n=250 m=1
accepted = True | source = ipopt:lower_25pct | U

,name,file,probtype,n,m_solver,integrality_relaxed,accepted,ub,source,max_bound_violation,max_constraint_violation,ipopt_attempts,ipopt_runtime,ipopt_status_summary,gurobi_repair_status,gurobi_repair_slack_sum,gurobi_repair_runtime
0,QPLIB_0018,QPLIB_0018.pkl,QCL,50,2,True,True,-6.386019,ipopt:clipped_zero,9.999521e-09,9.999217e-09,9,0.465332,"midpoint:status=0,obj=-6.3860189,feas=True,vio...",None,NaN,0.0
1,QPLIB_0031,QPLIB_0031.pkl,QML,60,33,True,True,15.200525,ipopt:clipped_zero,9.999999e-09,9.999671e-09,9,4.620625,"midpoint:status=0,obj=15.200525,feas=True,viol...",None,NaN,0.0
2,QPLIB_0032,QPLIB_0032.pkl,QML,100,53,True,True,9.945588,ipopt:clipped_zero,9.999997e-09,9.999497e-09,9,55.637327,"midpoint:status=1,obj=9.9455887,feas=True,viol...",None,NaN,0.0
3,QPLIB_0067,QPLIB_0067.pkl,QBL,80,1,True,True,-112205.360582,ipopt:clipped_zero,9.999997e-09,9.999894e-09,9,0.300198,"midpoint:status=0,obj=-112205.36,feas=True,vio...",None,NaN,0.0
4,QPLIB_0343,QPLIB_0343.pkl,QCL,50,2,True,True,-6.386019,ipopt:clipped_zero,9.999521e-09,9.999217e-09,9,0.470000,"midpoint:status=0,obj=-6.3860189,feas=True,vio...",None,NaN,0.0


In [7]:

# ============================================================
# 4. QPLIB B&B DATA GENERATOR
# Separate LP and SDP' trees.
# Sequential branching is intentionally preserved.
# ============================================================

@dataclass
class Node:
    node_id: int
    parent_id: Optional[int]
    depth: int
    l_bounds: np.ndarray
    u_bounds: np.ndarray
    branch_var: Optional[int] = None
    split_point: Optional[float] = None

    lp_obj: Optional[float] = None
    lp_time: Optional[float] = None
    lp_status: Optional[str] = None

    sdpp_obj: Optional[float] = None
    sdpp_time: Optional[float] = None
    sdpp_status: Optional[str] = None

    status: str = "pending"
    note: Optional[str] = None

    inherited_lb: Optional[float] = None
    known_lb: Optional[float] = None


def select_branch_variable(node: Node, min_width=MIN_WIDTH) -> Optional[int]:
    """
    Preserve the MINLPLib sequential rule.

    Start from depth % n.
    If that variable is fixed / too narrow, scan forward to the next
    branchable variable. We do NOT introduce a new branching heuristic here.
    """
    n = len(node.l_bounds)
    start = node.depth % n

    for offset in range(n):
        j = (start + offset) % n
        width = float(node.u_bounds[j] - node.l_bounds[j])

        if np.isfinite(width) and width > min_width:
            return int(j)

    return None


def create_children(parent: Node, branch_var: int, next_node_id: int):
    l = parent.l_bounds.copy()
    u = parent.u_bounds.copy()

    split = 0.5 * (float(l[branch_var]) + float(u[branch_var]))

    u_left = u.copy()
    u_left[branch_var] = split

    left = Node(
        node_id=next_node_id,
        parent_id=parent.node_id,
        depth=parent.depth + 1,
        l_bounds=l.copy(),
        u_bounds=u_left,
        branch_var=branch_var,
        split_point=split,
        inherited_lb=parent.known_lb,
        known_lb=parent.known_lb,
    )

    l_right = l.copy()
    l_right[branch_var] = split

    right = Node(
        node_id=next_node_id + 1,
        parent_id=parent.node_id,
        depth=parent.depth + 1,
        l_bounds=l_right,
        u_bounds=u.copy(),
        branch_var=branch_var,
        split_point=split,
        inherited_lb=parent.known_lb,
        known_lb=parent.known_lb,
    )

    return left, right


def solve_relaxation_at_node(data, node, relaxation):
    solver = QCQPSolver(
        A_list=data["A_list"],
        b_list=data["b_list"],
        c_list=data["c_list"],
        l=node.l_bounds,
        u=node.u_bounds,
        use_blocks=USE_BLOCKS,
        block_tol=BLOCK_TOL,
        verbose_blocks=VERBOSE_BLOCKS,
        prune_matrices=PRUNE_MATRICES,
        min_block_edges=MIN_BLOCK_EDGES,
    )

    if relaxation == "LP":
        if hasattr(solver, "solve_lp"):
            res = solver.solve_lp()
        else:
            res = solver.solve("LP")

    elif relaxation == "SDP_PRIME":
        if hasattr(solver, "solve_sdp_prime"):
            res = solver.solve_sdp_prime()
        else:
            res = solver.solve("SDP_PRIME")

    else:
        raise ValueError(relaxation)

    obj = res.get("objective")
    solve_time = res.get("time")
    solve_status = res.get("status")

    if relaxation == "LP":
        node.lp_obj = obj
        node.lp_time = solve_time
        node.lp_status = solve_status
    else:
        node.sdpp_obj = obj
        node.sdpp_time = solve_time
        node.sdpp_status = solve_status

    if solve_status is None:
        node.status = "error"
        node.note = "Missing solve status"
        return node

    s = str(solve_status).lower().strip()

    # This reproduces the MINLPLib B&B treatment.
    if s == "gurobi_status_4":
        node.status = "infeasible"
        node.note = (
            "Gurobi INF_OR_UNBD status 4 treated as infeasible "
            "because finite x bounds + McCormick envelopes make the LP node bounded."
        )
    elif "infeas" in s:
        node.status = "infeasible"
    elif "unknown" in s:
        node.status = "unknown"
        node.note = f"Unresolved solver status: {solve_status}"
    elif "unbound" in s:
        node.status = "unbounded"
        node.note = f"Unexpected unbounded solver status: {solve_status}"
    elif "error" in s or "exception" in s:
        node.status = "error"
        node.note = f"Solver error status: {solve_status}"
    elif "optimal" in s:
        node.status = "processed"
    else:
        node.status = "error"
        node.note = f"Unexpected solver status: {solve_status}"

    return node


def node_obj(node, relaxation):
    return node.lp_obj if relaxation == "LP" else node.sdpp_obj


def compute_global_frontier_lb(queue, open_terminal_nodes, closed_by_bound_lbs):
    represented_lbs = []

    for active_node in list(queue) + list(open_terminal_nodes.values()):
        if active_node.known_lb is None:
            return np.nan

        lb = float(active_node.known_lb)

        if np.isnan(lb):
            return np.nan

        represented_lbs.append(lb)

    represented_lbs.extend(float(lb) for lb in closed_by_bound_lbs)

    if represented_lbs:
        return float(min(represented_lbs))

    # Empty represented tree: do not substitute the incumbent UB as a lower bound.
    return np.nan


def run_tree(
    data,
    relaxation,
    shared_upper_bound,
    max_nodes,
    live_csv_path,
    instance_index,
    ub_info,
):
    node_counter = 0
    processed_count = 0
    had_unresolved_status = False

    open_terminal_nodes = {}
    closed_by_bound_lbs = []

    root = Node(
        node_id=node_counter,
        parent_id=None,
        depth=0,
        l_bounds=data["l"].copy(),
        u_bounds=data["u"].copy(),
    )
    node_counter += 1

    queue = deque([root])
    node_buffer = []
    all_nodes = []
    last_frontier_lb = np.nan

    def flush():
        nonlocal node_buffer

        if not node_buffer:
            return

        df = pd.DataFrame(node_buffer)
        exists = live_csv_path.exists()

        df.to_csv(
            live_csv_path,
            mode="a",
            header=not exists,
            index=False,
        )
        node_buffer = []

    while queue:
        if max_nodes is not None and processed_count >= max_nodes:
            break

        node = queue.popleft()

        try:
            node = solve_relaxation_at_node(data, node, relaxation)
        except Exception as e:
            node.status = "error"
            node.note = f"{type(e).__name__}: {e}"

        all_nodes.append(node)
        processed_count += 1

        obj = node_obj(node, relaxation)

        if node.status == "processed":
            if obj is None or not np.isfinite(float(obj)):
                node.status = "error"
                node.note = "Processed node has no finite relaxation objective."
            else:
                node.known_lb = float(obj)

        stop_tree = False

        if node.status in {"unknown", "unbounded", "error"}:
            open_terminal_nodes[node.node_id] = node
            had_unresolved_status = True
            stop_tree = True

        elif node.status == "infeasible":
            # Proven empty region.
            pass

        elif (
            node.known_lb is not None
            and np.isfinite(float(node.known_lb))
            and float(node.known_lb) >= float(shared_upper_bound) - PRUNE_TOL
        ):
            node.status = "pruned_by_bound"
            node.note = (
                f"LB={float(node.known_lb):.12g} >= "
                f"UB={float(shared_upper_bound):.12g} - tol={PRUNE_TOL:.3g}"
            )
            closed_by_bound_lbs.append(float(node.known_lb))

        elif node.depth >= MAX_DEPTH:
            open_terminal_nodes[node.node_id] = node

        else:
            branch_var = select_branch_variable(node, min_width=MIN_WIDTH)

            if branch_var is None:
                node.status = "pruned"
                node.note = "No branchable variable with positive finite width."
                open_terminal_nodes[node.node_id] = node

            else:
                left, right = create_children(
                    parent=node,
                    branch_var=branch_var,
                    next_node_id=node_counter,
                )
                node_counter += 2
                queue.append(left)
                queue.append(right)

        last_frontier_lb = compute_global_frontier_lb(
            queue,
            open_terminal_nodes,
            closed_by_bound_lbs,
        )

        row = {
            "dataset_name": "qplib_fixed100",
            "instance_index": instance_index,
            "instance_name": data["name"],
            "instance_id": data["instance_id"],
            "probtype": data["probtype"],
            "integrality_relaxed": data["integrality_relaxed"],
            "convex": data["convex"],
            "n": data["n"],
            "m_original": data["m_original"],
            "m_solver": data["m_solver"],

            "relaxation": relaxation,
            "node_id": node.node_id,
            "parent_id": node.parent_id,
            "depth": node.depth,
            "branch_var": node.branch_var,
            "split_point": node.split_point,

            "status": node.status,
            "note": node.note,
            "solve_status": (
                node.lp_status if relaxation == "LP" else node.sdpp_status
            ),
            "obj": obj,
            "solve_time": (
                node.lp_time if relaxation == "LP" else node.sdpp_time
            ),

            "processed_count": processed_count,
            "inherited_lb": node.inherited_lb,
            "known_lb": node.known_lb,
            "shared_upper_bound": shared_upper_bound,
            "ub_source": ub_info.get("source"),
            "ub_max_bound_violation": ub_info.get("max_bound_violation"),
            "ub_max_constraint_violation": ub_info.get("max_constraint_violation"),

            "global_frontier_lb_after_step": last_frontier_lb,
            "queue_count_after_step": len(queue),
            "open_terminal_count_after_step": len(open_terminal_nodes),
            "pruned_by_bound": node.status == "pruned_by_bound",

            "lp_node_budget": FIXED_NODES_LP,
            "sdpp_node_budget": FIXED_NODES_SDPP,
            "branch_rule": BRANCH_RULE,
            "split_rule": SPLIT_RULE,
            "coefficient_convention": "0.5*xTAx + bTx + c",
        }

        node_buffer.append(row)

        if len(node_buffer) >= FLUSH_EVERY_NODES:
            flush()

        if stop_tree:
            flush()
            break

    flush()

    return {
        "nodes": all_nodes,
        "had_unresolved_status": had_unresolved_status,
        "processed_nodes": processed_count,
        "final_frontier_lb": last_frontier_lb,
        "queue_remaining": len(queue),
    }


def clean_partial_bb_rows_if_resuming():
    """
    Completed instances are listed in QPLIB_BB_SUMMARY_CSV.

    If a crash occurred in the middle of an instance, remove LIVE rows for
    names not present in the completed summary before resuming.
    """
    if not RESUME_BB:
        return set()

    completed = set()

    if QPLIB_BB_SUMMARY_CSV.exists():
        summary = pd.read_csv(QPLIB_BB_SUMMARY_CSV)
        if "instance_name" in summary.columns:
            completed = set(summary["instance_name"].astype(str))

    if QPLIB_LIVE_CSV.exists():
        live = pd.read_csv(QPLIB_LIVE_CSV)

        if "instance_name" in live.columns:
            clean = live[live["instance_name"].astype(str).isin(completed)].copy()

            if len(clean) != len(live):
                print(
                    f"Resume cleanup: removed {len(live)-len(clean)} "
                    "partial LIVE rows from incomplete instances."
                )
                clean.to_csv(QPLIB_LIVE_CSV, index=False)

    return completed


def run_qplib_bb(pkl_files, feasible_points):
    QPLIB_BB_DIR.mkdir(parents=True, exist_ok=True)

    completed = clean_partial_bb_rows_if_resuming()

    if not RESUME_BB:
        for path in [
            QPLIB_LIVE_CSV,
            QPLIB_LIVE_PARQUET,
            QPLIB_BB_SUMMARY_CSV,
            QPLIB_BB_SKIPPED_CSV,
        ]:
            if path.exists():
                path.unlink()

    summary_rows = []
    skipped_rows = []

    if QPLIB_BB_SUMMARY_CSV.exists() and RESUME_BB:
        summary_rows = pd.read_csv(QPLIB_BB_SUMMARY_CSV).to_dict("records")

    if QPLIB_BB_SKIPPED_CSV.exists() and RESUME_BB:
        skipped_rows = pd.read_csv(QPLIB_BB_SKIPPED_CSV).to_dict("records")

    for idx, pkl_path in enumerate(pkl_files):
        data = load_qplib_pickle(pkl_path)
        name = str(data["name"])

        if RESUME_BB and name in completed:
            print(f"[{idx+1}/{len(pkl_files)}] {name}: already complete -> skip")
            continue

        print(
            f"\n[{idx+1}/{len(pkl_files)}] {name} "
            f"| n={data['n']} m={data['m_solver']}"
        )

        ub_info = feasible_points.get(name)

        if (
            ub_info is None
            or ub_info.get("x") is None
            or ub_info.get("ub") is None
            or not np.isfinite(float(ub_info["ub"]))
        ):
            skipped_rows.append({
                "instance_name": name,
                "file": pkl_path.name,
                "reason": "no_verified_feasible_incumbent",
            })
            pd.DataFrame(skipped_rows).drop_duplicates(
                subset=["instance_name"], keep="last"
            ).to_csv(QPLIB_BB_SKIPPED_CSV, index=False)

            print("SKIPPED: no verified feasible incumbent.")
            continue

        # Revalidate before using the point for pruning.
        check = check_feasibility(data, ub_info["x"])

        if not check["is_feasible"]:
            skipped_rows.append({
                "instance_name": name,
                "file": pkl_path.name,
                "reason": "stored_incumbent_failed_revalidation",
                "max_bound_violation": check["max_bound_violation"],
                "max_constraint_violation": check["max_constraint_violation"],
            })
            pd.DataFrame(skipped_rows).drop_duplicates(
                subset=["instance_name"], keep="last"
            ).to_csv(QPLIB_BB_SKIPPED_CSV, index=False)

            print("SKIPPED: incumbent failed revalidation.")
            continue

        shared_ub = objective_value(data, ub_info["x"])

        print(
            f"Incumbent UB={shared_ub:.12g} "
            f"| source={ub_info.get('source')}"
        )

        lp = run_tree(
            data=data,
            relaxation="LP",
            shared_upper_bound=shared_ub,
            max_nodes=FIXED_NODES_LP,
            live_csv_path=QPLIB_LIVE_CSV,
            instance_index=idx,
            ub_info=ub_info,
        )

        if lp["had_unresolved_status"]:
            summary_rows.append({
                "instance_name": name,
                "probtype": data["probtype"],
                "n": data["n"],
                "m_solver": data["m_solver"],
                "ub": shared_ub,
                "ub_source": ub_info.get("source"),
                "lp_nodes": lp["processed_nodes"],
                "lp_final_frontier_lb": lp["final_frontier_lb"],
                "lp_unresolved": True,
                "sdpp_nodes": 0,
                "sdpp_final_frontier_lb": np.nan,
                "sdpp_unresolved": np.nan,
                "complete": True,
                "usable_for_pair": False,
            })

            pd.DataFrame(summary_rows).drop_duplicates(
                subset=["instance_name"], keep="last"
            ).to_csv(QPLIB_BB_SUMMARY_CSV, index=False)

            print("LP unresolved -> SDP' skipped, matching MINLPLib policy.")
            continue

        sdpp = run_tree(
            data=data,
            relaxation="SDP_PRIME",
            shared_upper_bound=shared_ub,
            max_nodes=FIXED_NODES_SDPP,
            live_csv_path=QPLIB_LIVE_CSV,
            instance_index=idx,
            ub_info=ub_info,
        )

        summary_rows.append({
            "instance_name": name,
            "probtype": data["probtype"],
            "n": data["n"],
            "m_solver": data["m_solver"],
            "integrality_relaxed": data["integrality_relaxed"],
            "convex": data["convex"],

            "ub": shared_ub,
            "ub_source": ub_info.get("source"),

            "lp_nodes": lp["processed_nodes"],
            "lp_final_frontier_lb": lp["final_frontier_lb"],
            "lp_unresolved": lp["had_unresolved_status"],

            "sdpp_nodes": sdpp["processed_nodes"],
            "sdpp_final_frontier_lb": sdpp["final_frontier_lb"],
            "sdpp_unresolved": sdpp["had_unresolved_status"],

            "complete": True,
            "usable_for_pair": (
                not lp["had_unresolved_status"]
                and not sdpp["had_unresolved_status"]
                and np.isfinite(lp["final_frontier_lb"])
                and np.isfinite(sdpp["final_frontier_lb"])
            ),
        })

        summary_df = pd.DataFrame(summary_rows).drop_duplicates(
            subset=["instance_name"], keep="last"
        )
        summary_df.to_csv(QPLIB_BB_SUMMARY_CSV, index=False)

        print(
            "LP final LB =", lp["final_frontier_lb"],
            "| SDP' final LB =", sdpp["final_frontier_lb"],
            "| usable =", not sdpp["had_unresolved_status"],
        )

        gc.collect()

    summary_df = pd.DataFrame(summary_rows).drop_duplicates(
        subset=["instance_name"], keep="last"
    ).reset_index(drop=True)

    if QPLIB_LIVE_CSV.exists():
        try:
            pd.read_csv(QPLIB_LIVE_CSV).to_parquet(
                QPLIB_LIVE_PARQUET,
                index=False,
            )
            print("Saved:", QPLIB_LIVE_PARQUET)
        except Exception as e:
            print("[WARN] parquet conversion failed:", repr(e))

    print("\nQPLIB B&B instance summary")
    if not summary_df.empty:
        print("Completed:", len(summary_df))
        print(
            "Usable LP-SDP' pair:",
            int(summary_df["usable_for_pair"].fillna(False).sum()),
        )

    print("Summary:", QPLIB_BB_SUMMARY_CSV)
    print("LIVE   :", QPLIB_LIVE_CSV)

    return summary_df


# RUN THIS CELL TO GENERATE / RESUME THE FIXED-BUDGET QPLIB B&B DATASET.
qplib_bb_summary = run_qplib_bb(
    pkl_files=pkl_files,
    feasible_points=qplib_feasible_points,
)

display(qplib_bb_summary.head())



[1/143] QPLIB_0018 | n=50 m=2
Incumbent UB=-6.38601886211 | source=ipopt:clipped_zero
LP final LB = -137.86797112222234 | SDP' final LB = -89.93953439946289 | usable = True

[2/143] QPLIB_0031 | n=60 m=33
Incumbent UB=15.2005247256 | source=ipopt:clipped_zero
LP final LB = 0.0 | SDP' final LB = -25794.93718091263 | usable = True

[3/143] QPLIB_0032 | n=100 m=53
Incumbent UB=9.94558798396 | source=ipopt:clipped_zero
LP final LB = 0.0 | SDP' final LB = -15552.090905318984 | usable = True

[4/143] QPLIB_0067 | n=80 m=1
Incumbent UB=-112205.360582 | source=ipopt:clipped_zero
LP final LB = -112216.38515901056 | SDP' final LB = -116136.41370447619 | usable = True

[5/143] QPLIB_0343 | n=50 m=2
Incumbent UB=-6.38601886211 | source=ipopt:clipped_zero
LP final LB = -137.86793001111124 | SDP' final LB = -89.93954051761112 | usable = True

[6/143] QPLIB_0633 | n=75 m=2
Incumbent UB=79.5791682108 | source=ipopt:random_0
LP final LB = 2.952142351425 | SDP' final LB = 73.06740407740216 | usable = T

,instance_name,probtype,n,m_solver,integrality_relaxed,convex,ub,ub_source,lp_nodes,lp_final_frontier_lb,lp_unresolved,sdpp_nodes,sdpp_final_frontier_lb,sdpp_unresolved,complete,usable_for_pair
0,QPLIB_0018,QCL,50,2,True,None,-6.386019,ipopt:clipped_zero,100,-137.867971,False,100,-89.939534,False,True,True
1,QPLIB_0031,QML,60,33,True,None,15.200525,ipopt:clipped_zero,100,0.000000,False,100,-25794.937181,False,True,True
2,QPLIB_0032,QML,100,53,True,None,9.945588,ipopt:clipped_zero,100,0.000000,False,100,-15552.090905,False,True,True
3,QPLIB_0067,QBL,80,1,True,None,-112205.360582,ipopt:clipped_zero,100,-112216.385159,False,100,-116136.413704,False,True,True
4,QPLIB_0343,QCL,50,2,True,None,-6.386019,ipopt:clipped_zero,100,-137.867930,False,100,-89.939541,False,True,True


In [8]:

# ============================================================
# 5. QPLIB B&B QUICK TARGET SUMMARY
# This is ONLY the initial fixed-100-node tree dataset.
# Final thesis Method 1 / Method 2 time-aware budgets come later.
# ============================================================

def compute_delta(z_lp, z_other):
    if not (np.isfinite(z_lp) and np.isfinite(z_other)):
        return np.nan

    return (
        (float(z_lp) - float(z_other))
        /
        (abs(float(z_lp)) + abs(float(z_other)) + ETA)
    )


if not qplib_bb_summary.empty:
    qplib_bb_summary = qplib_bb_summary.copy()

    qplib_bb_summary["delta_lp_sdpp_fixed100"] = qplib_bb_summary.apply(
        lambda r: compute_delta(
            r["lp_final_frontier_lb"],
            r["sdpp_final_frontier_lb"],
        )
        if bool(r.get("usable_for_pair", False))
        else np.nan,
        axis=1,
    )

    qplib_bb_summary["label_lp_sdpp_fixed100"] = qplib_bb_summary[
        "delta_lp_sdpp_fixed100"
    ].apply(
        lambda d: int(d > -EPS) if np.isfinite(d) else np.nan
    )

    qplib_bb_summary.to_csv(QPLIB_BB_SUMMARY_CSV, index=False)

    print(
        qplib_bb_summary["label_lp_sdpp_fixed100"]
        .value_counts(dropna=False)
        .rename(index={1.0: "LP/near-tie", 0.0: "SDP' clearly stronger"})
    )

    display(
        qplib_bb_summary[
            [
                "instance_name",
                "probtype",
                "n",
                "ub",
                "lp_nodes",
                "sdpp_nodes",
                "lp_final_frontier_lb",
                "sdpp_final_frontier_lb",
                "delta_lp_sdpp_fixed100",
                "label_lp_sdpp_fixed100",
                "usable_for_pair",
            ]
        ].head(20)
    )


label_lp_sdpp_fixed100
SDP' clearly stronger    88
LP/near-tie              48
NaN                       4
Name: count, dtype: int64


,instance_name,probtype,n,ub,lp_nodes,sdpp_nodes,lp_final_frontier_lb,sdpp_final_frontier_lb,delta_lp_sdpp_fixed100,label_lp_sdpp_fixed100,usable_for_pair
0,QPLIB_0018,QCL,50,-6.386019e+00,100,100,-137.867971,-8.993953e+01,-2.103891e-01,0.0,True
1,QPLIB_0031,QML,60,1.520052e+01,100,100,0.000000,-2.579494e+04,1.000000e+00,1.0,True
2,QPLIB_0032,QML,100,9.945588e+00,100,100,0.000000,-1.555209e+04,9.999999e-01,1.0,True
3,QPLIB_0067,QBL,80,-1.122054e+05,100,100,-112216.385159,-1.161364e+05,1.716654e-02,1.0,True
4,QPLIB_0343,QCL,50,-6.386019e+00,100,100,-137.867930,-8.993954e+01,-2.103890e-01,0.0,True
5,QPLIB_0633,QBL,75,7.957917e+01,100,100,2.952142,7.306740e+01,-9.223199e-01,0.0,True
6,QPLIB_0681,LMQ,215,-4.826343e+01,100,100,-48.390600,-4.839060e+01,-1.794357e-10,1.0,True
7,QPLIB_0682,LMQ,261,-4.804304e+01,100,100,-48.390600,-4.839060e+01,-1.978078e-10,1.0,True
8,QPLIB_0698,LMQ,118,9.685721e+05,100,100,603634.460576,3.435483e+05,2.745893e-01,1.0,True
9,QPLIB_0752,QBL,250,-2.402200e+04,100,100,-39610.500000,-2.543467e+04,-2.179383e-01,0.0,True
